In [1]:
#Load packages
import pandas as pd
import numpy as np

# I. Coding Run Expectancy Dataset (2017)

In [34]:
# Read in MLBAM Data for 2017

MLBAM17 = pd.read_csv("../Data/MLBAM17.csv")

In [35]:
#Your Code Here
# Here we create a df RE18 which includes only columns we need to calculate the run expectancy matrix.

RE17 = MLBAM17[['batterName','batterId','event', 'start1B', 'start2B', 'start3B', 'end1B', 'end2B', 'end3B',\
                   'startOuts','endOuts','runsFuture','runsOnPlay','outsInInning',\
                   'stand', 'throws','venueId', 'stadium', 'batterPos']].copy()

display(RE17)                   

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,endOuts,runsFuture,runsOnPlay,outsInInning,stand,throws,venueId,stadium,batterPos
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,3,L,R,12,Tropicana Field,LF
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,2,0,0,3,R,R,12,Tropicana Field,C
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,2,0,0,3,L,R,12,Tropicana Field,1B
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,3,0,0,3,R,R,12,Tropicana Field,DH
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,0,3,0,3,L,R,12,Tropicana Field,DH
5,Kiermaier,595281,Double,572816.0,NaN,NaN,NaN,595281.0,572816.0,0,0,3,0,3,L,R,12,Tropicana Field,CF
6,Longoria,446334,Sac Fly,NaN,595281.0,572816.0,NaN,595281.0,NaN,0,1,3,1,3,R,R,12,Tropicana Field,3B
7,"Miller, B",543543,Single,NaN,595281.0,NaN,543543.0,NaN,595281.0,1,1,2,0,3,L,R,12,Tropicana Field,2B
8,Souza Jr.,519306,Walk,543543.0,NaN,595281.0,519306.0,543543.0,595281.0,1,1,2,0,3,R,R,12,Tropicana Field,RF
9,Morrison,489149,Single,519306.0,543543.0,595281.0,489149.0,519306.0,NaN,1,1,2,2,3,L,R,12,Tropicana Field,1B


In [36]:
RE17['Start1'] = np.where(pd.isnull(RE17['start1B']),0,1)
RE17['Start2'] = np.where(pd.isnull(RE17['start2B']),0,1)
RE17['Start3'] = np.where(pd.isnull(RE17['start3B']),0,1)

RE17['Start_State'] = (RE17['Start1'].astype(str) + RE17['Start2'].astype(str) + RE17['Start3'].astype(str)+\
                          " " + RE17['startOuts'].astype(str))

RE17['End1'] = np.where(pd.isnull(RE17['end1B']),0,1)
RE17['End2'] = np.where(pd.isnull(RE17['end2B']),0,1)
RE17['End3'] = np.where(pd.isnull(RE17['end3B']),0,1)

RE17['End_State'] = (RE17['End1'].astype(str) + RE17['End2'].astype(str) + RE17['End3'].astype(str) + \
                        " " + RE17['endOuts'].astype(str))

RE17 = RE17[((RE17.Start_State != RE17.End_State) | (RE17.runsOnPlay > 0)) & (RE17.outsInInning == 3)]

Start_RunExp = RE17.groupby(['Start_State'])['runsFuture'].mean().reset_index().rename(columns={'runsFuture':'Start_RE'})

RE17 = pd.merge(RE17, Start_RunExp, on=['Start_State'], how='left')

Base_State_3 = [pd.Series(['000 3', 0], index=Start_RunExp.columns),
                pd.Series(['001 3', 0], index=Start_RunExp.columns),
                pd.Series(['010 3', 0], index=Start_RunExp.columns),
                pd.Series(['011 3', 0], index=Start_RunExp.columns),
                pd.Series(['100 3', 0], index=Start_RunExp.columns),
                pd.Series(['101 3', 0], index=Start_RunExp.columns),
                pd.Series(['110 3', 0], index=Start_RunExp.columns),
                pd.Series(['111 3', 0], index=Start_RunExp.columns)]
Start_RunExp = Start_RunExp.append(Base_State_3, ignore_index=True)

End_RunExp = Start_RunExp.rename(columns={'Start_State':'End_State', 'Start_RE':'End_RE'})

RE17 = pd.merge(RE17, End_RunExp, on=['End_State'], how='left')

RE17['Run_Value'] = RE17['runsOnPlay'] + RE17['End_RE'] - RE17['Start_RE']

RE17

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,...,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE,End_RE,Run_Value
0,Gardner,458731,Flyout,NaN,NaN,NaN,NaN,NaN,NaN,0,...,0,0,000 0,0,0,0,000 1,0.516375,0.272176,-0.244199
1,"Sanchez, G",596142,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,...,0,0,000 1,0,0,0,000 2,0.272176,0.108038,-0.164138
2,Bird,595885,Walk,NaN,NaN,NaN,595885.0,NaN,NaN,2,...,0,0,000 2,1,0,0,100 2,0.108038,0.225365,0.117327
3,Holliday,407812,Groundout,595885.0,NaN,NaN,NaN,NaN,NaN,2,...,0,0,100 2,0,0,0,000 3,0.225365,0.000000,-0.225365
4,"Dickerson, C",572816,Single,NaN,NaN,NaN,572816.0,NaN,NaN,0,...,0,0,000 0,1,0,0,100 0,0.516375,0.912921,0.396546
5,Kiermaier,595281,Double,572816.0,NaN,NaN,NaN,595281.0,572816.0,0,...,0,0,100 0,0,1,1,011 0,0.912921,2.051376,1.138455
6,Longoria,446334,Sac Fly,NaN,595281.0,572816.0,NaN,595281.0,NaN,0,...,1,1,011 0,0,1,0,010 1,2.051376,0.711650,-0.339727
7,"Miller, B",543543,Single,NaN,595281.0,NaN,543543.0,NaN,595281.0,1,...,1,0,010 1,1,0,1,101 1,0.711650,1.215306,0.503657
8,Souza Jr.,519306,Walk,543543.0,NaN,595281.0,519306.0,543543.0,595281.0,1,...,0,1,101 1,1,1,1,111 1,1.215306,1.635608,0.420302
9,Morrison,489149,Single,519306.0,543543.0,595281.0,489149.0,519306.0,NaN,1,...,1,1,111 1,1,1,0,110 1,1.635608,0.947424,1.311816


In [39]:
Event_Value17 = RE17.groupby(['event'])['Run_Value'].mean().reset_index()
Event_Value17 = Event_Value17.rename(columns={'Run_Value':'Run_Value17'})
Event_Value17

,event,Run_Value17
0,Batter Interference,-0.430019
1,Bunt Groundout,-0.209411
2,Bunt Lineout,-0.328292
3,Bunt Pop Out,-0.373225
4,Catcher Interference,0.399070
5,Double,0.779338
6,Double Play,-0.897164
7,Fan interference,0.590743
8,Field Error,0.493206
9,Fielders Choice,0.764112


# II. Coding Run Expectancy Dataset (2016)

In [40]:
# Read in MLBAM Data for 2016

MLBAM16 = pd.read_csv("../Data/MLBAM16.csv")

In [41]:
#Your Code Here
RE16 = MLBAM16[['batterName','batterId','event', 'start1B', 'start2B', 'start3B', 'end1B', 'end2B', 'end3B',\
                   'startOuts','endOuts','runsFuture','runsOnPlay','outsInInning',\
                   'stand', 'throws','venueId', 'stadium', 'batterPos']].copy()

RE16['Start1'] = np.where(pd.isnull(RE16['start1B']),0,1)
RE16['Start2'] = np.where(pd.isnull(RE16['start2B']),0,1)
RE16['Start3'] = np.where(pd.isnull(RE16['start3B']),0,1)

RE16['Start_State'] = (RE16['Start1'].astype(str) + RE16['Start2'].astype(str) + RE16['Start3'].astype(str)+\
                          " " + RE16['startOuts'].astype(str))

RE16['End1'] = np.where(pd.isnull(RE16['end1B']),0,1)
RE16['End2'] = np.where(pd.isnull(RE16['end2B']),0,1)
RE16['End3'] = np.where(pd.isnull(RE16['end3B']),0,1)

RE16['End_State'] = (RE16['End1'].astype(str) + RE16['End2'].astype(str) + RE16['End3'].astype(str) + \
                        " " + RE16['endOuts'].astype(str))

RE16 = RE16[((RE16.Start_State != RE16.End_State) | (RE16.runsOnPlay > 0)) & (RE16.outsInInning == 3)]

Start_RunExp = RE16.groupby(['Start_State'])['runsFuture'].mean().reset_index().rename(columns={'runsFuture':'Start_RE'})

RE16 = pd.merge(RE16, Start_RunExp, on=['Start_State'], how='left')

Base_State_3 = [pd.Series(['000 3', 0], index=Start_RunExp.columns),
                pd.Series(['001 3', 0], index=Start_RunExp.columns),
                pd.Series(['010 3', 0], index=Start_RunExp.columns),
                pd.Series(['011 3', 0], index=Start_RunExp.columns),
                pd.Series(['100 3', 0], index=Start_RunExp.columns),
                pd.Series(['101 3', 0], index=Start_RunExp.columns),
                pd.Series(['110 3', 0], index=Start_RunExp.columns),
                pd.Series(['111 3', 0], index=Start_RunExp.columns)]
Start_RunExp = Start_RunExp.append(Base_State_3, ignore_index=True)

End_RunExp = Start_RunExp.rename(columns={'Start_State':'End_State', 'Start_RE':'End_RE'})

RE16 = pd.merge(RE16, End_RunExp, on=['End_State'], how='left')

RE16['Run_Value'] = RE16['runsOnPlay'] + RE16['End_RE'] - RE16['Start_RE']

RE16

,batterName,batterId,event,start1B,start2B,start3B,end1B,end2B,end3B,startOuts,...,Start2,Start3,Start_State,End1,End2,End3,End_State,Start_RE,End_RE,Run_Value
0,"Carpenter, M",572761,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,0,...,0,0,000 0,0,0,0,000 1,0.498377,0.268678,-0.229699
1,Pham,502054,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,1,...,0,0,000 1,0,0,0,000 2,0.268678,0.106305,-0.162373
2,Holliday,407812,Strikeout,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,0,000 2,0,0,0,000 3,0.106305,0.000000,-0.106305
3,Jaso,444379,Groundout,NaN,NaN,NaN,NaN,NaN,NaN,0,...,0,0,000 0,0,0,0,000 1,0.498377,0.268678,-0.229699
4,McCutchen,457705,Hit By Pitch,NaN,NaN,NaN,457705.0,NaN,NaN,1,...,0,0,000 1,1,0,0,100 1,0.268678,0.512225,0.243547
5,Freese,501896,Single,457705.0,NaN,NaN,501896.0,NaN,457705.0,1,...,0,0,100 1,1,0,1,101 1,0.512225,1.196777,0.684552
6,"Marte, S",516782,Lineout,501896.0,NaN,457705.0,501896.0,NaN,457705.0,1,...,0,1,101 1,1,0,1,101 2,1.196777,0.480175,-0.716603
7,Cervelli,465041,Pop Out,501896.0,NaN,457705.0,NaN,NaN,NaN,2,...,0,1,101 2,0,0,0,000 3,0.480175,0.000000,-0.480175
8,Grichuk,545341,Strikeout,NaN,NaN,NaN,NaN,NaN,NaN,0,...,0,0,000 0,0,0,0,000 1,0.498377,0.268678,-0.229699
9,Piscotty,572039,Strikeout,NaN,NaN,NaN,NaN,NaN,NaN,1,...,0,0,000 1,0,0,0,000 2,0.268678,0.106305,-0.162373


In [42]:
Event_Value16 = RE16.groupby(['event'])['Run_Value'].mean().reset_index()
Event_Value16 = Event_Value16.rename(columns={'Run_Value':'Run_Value16'})
Event_Value16

,event,Run_Value16
0,Batter Interference,-0.284649
1,Bunt Groundout,-0.218826
2,Bunt Lineout,-0.352295
3,Bunt Pop Out,-0.342802
4,Catcher Interference,0.301623
5,Double,0.743467
6,Double Play,-0.864981
7,Fan interference,0.533316
8,Field Error,0.469989
9,Fielders Choice,0.701447


# III. Comparing 2016 vs. 2017

In [43]:
#Your Code Here
Event_1617 = pd.merge(Event_Value16,Event_Value17,on=['event'])
Event_1617

,event,Run_Value16,Run_Value17
0,Batter Interference,-0.284649,-0.430019
1,Bunt Groundout,-0.218826,-0.209411
2,Bunt Lineout,-0.352295,-0.328292
3,Bunt Pop Out,-0.342802,-0.373225
4,Catcher Interference,0.301623,0.399070
5,Double,0.743467,0.779338
6,Double Play,-0.864981,-0.897164
7,Fan interference,0.533316,0.590743
8,Field Error,0.469989,0.493206
9,Fielders Choice,0.701447,0.764112


In [51]:
Event_1617['diff'] = Event_1617['Run_Value17'] - Event_1617['Run_Value16']
Event_1617.sort_values(by='diff')

,event,Run_Value16,Run_Value17,diff
23,Sac Fly DP,-0.370443,-0.543715,-0.173272
0,Batter Interference,-0.284649,-0.430019,-0.145369
26,Strikeout - DP,-0.622787,-0.682819,-0.060032
22,Sac Fly,0.023233,-0.032119,-0.055352
10,Fielders Choice Out,-0.610757,-0.662400,-0.051643
13,Grounded Into DP,-0.773318,-0.818005,-0.044687
6,Double Play,-0.864981,-0.897164,-0.032183
3,Bunt Pop Out,-0.342802,-0.373225,-0.030423
12,Forceout,-0.326657,-0.345347,-0.018691
20,Runner Out,-0.257904,-0.274991,-0.017087


In [47]:
Player_Value16 = RE16.groupby(['batterName'])['Run_Value'].sum().reset_index()
Player_Value16 = Player_Value16.rename(columns={'Run_Value':'Run_Value16'})
# Player_Value16 = Player_Value16.sort_values(by=['Run_Value'],  ascending=False) 
Player_Value17 = RE17.groupby(['batterName'])['Run_Value'].sum().reset_index()
Player_Value17 = Player_Value17.rename(columns={'Run_Value':'Run_Value17'})
# Player_Value17 = Player_Value17.sort_values(by=['Run_Value'],  ascending=False)
Player_1617 = pd.merge(Player_Value16,Player_Value17,on=['batterName'])

In [49]:
Player_1617.sort_values(by='Run_Value16',ascending=True)

,batterName,Run_Value16,Run_Value17
269,Hechavarria,-32.142529,-6.378460
440,"Norris, De",-26.841250,-11.798153
171,"Escobar, A",-26.555530,-22.348618
78,Burns,-23.335961,-1.429560
6,Ahmed,-23.000581,-4.176089
683,Zimmerman,-22.942623,33.179676
201,Galvis,-22.751256,-13.695274
370,"Marte, K",-21.742476,-3.503534
296,"Iglesias, J",-21.271729,-11.173195
309,"Joseph, Ca",-20.085234,-8.346937


In [52]:
Player_1617['diff'] = Player_1617['Run_Value17'] - Player_1617['Run_Value16']
Player_1617.sort_values(by='diff')

,batterName,Run_Value16,Run_Value17,diff
449,Ortiz,60.938730,-2.629789,-63.568519
498,"Ramirez, H",30.676190,-19.461039,-50.137229
444,Odor,11.679587,-36.464351,-48.143938
82,Cabrera,37.520249,-4.336300,-41.856549
637,Trumbo,21.644283,-17.990412,-39.634695
47,Beltran,21.279393,-16.883042,-38.162435
384,Maybin,23.263283,-12.318532,-35.581815
572,Segura,28.717694,-5.523280,-34.240974
227,"Gonzalez, C",28.576921,-5.394397,-33.971317
160,Duvall,21.819293,-11.639746,-33.459040
